In [1]:
import os
import sys
import pprint
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import SelectorGroupChat, RoundRobinGroupChat
from autogen_agentchat.tools import AgentTool
from autogen_agentchat.conditions import MaxMessageTermination, TextMentionTermination
from autogen_agentchat.ui import Console
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_ext.tools.mcp import McpWorkbench, StdioServerParams, mcp_server_tools

# Project import paths
# Get the current directory (which is project/test)
notebook_dir = os.getcwd()

# Compute absolute paths for lib and src (one level up from test/)
lib_path = os.path.abspath(os.path.join(notebook_dir, '..', 'lib'))
src_path = os.path.abspath(os.path.join(notebook_dir, '..', 'src'))

sys.path.append(lib_path)
sys.path.append(src_path)

from model_loader import load_model_client

In [2]:
model_client = load_model_client("azure")

In [3]:
# Helpful sanity prints:
print("raw:", getattr(model_client, "_raw_config", None))
print("create_args:", getattr(model_client, "_create_args", None))

raw: {'model': 'gpt-4o', 'api_key': 'REPLACE_WITH_YOUR_API_KEY', 'azure_endpoint': 'https://224d2-m9kjdmy0-eastus2.openai.azure.com/', 'azure_deployment': 'gpt-4o', 'api_version': '2024-12-01-preview'}
create_args: {'model': 'gpt-4o'}


In [4]:
PyMCP_FS = StdioServerParams(command="/run/media/jun/Jun_6T/Code/mcp_servers/PyMCP-FS/.venv/bin/python", args=[
    		"/run/media/jun/Jun_6T/Code/mcp_servers/PyMCP-FS/main.py","-d","/run/media/jun/Jun_6T/Code/TFM_BA_GPT/",	
    		"/tmp/",
            "--log-level",
            "DEBUG",
            "--log-file",
            "mcp_debug.log"])

Mermaid_MCP = StdioServerParams(command="/run/media/jun/Jun_6T/Code/mcp_servers/Mermaid-MCP/venv/bin/python", args=[
				"/run/media/jun/Jun_6T/Code/mcp_servers/Mermaid-MCP/Mermaid-MCP.py",
				"-d",
				"/run/media/jun/Jun_6T/Code/TFM_BA_GPT/",
				"-d",
				"/tmp/"])

PDF_format_mcp_server = StdioServerParams(command="/run/media/jun/Jun_6T/Code/mcp_servers/Final_deliverable_MCP_server/venv/bin/python", args=[
    					"/run/media/jun/Jun_6T/Code/mcp_servers/Final_deliverable_MCP_server/PDF_format_mcp_server.py", 
                        "-d", 
                        "/run/media/jun/Jun_6T/Code/TFM_BA_GPT/experience"])

PyMCP_FS_mcp_tools = await mcp_server_tools(PyMCP_FS)
mermaid_mcp_tools = await mcp_server_tools(Mermaid_MCP)
pdf_format_mcp_tools = await mcp_server_tools(PDF_format_mcp_server)

In [ ]:

# ---- terminate_agent system message (MOVE + clean + render_mermaid_all + robust PDF) ----
terminate_system_message = r"""
You are the TERMINATE_AGENT. Perform the final export pipeline and then end with **TERMINATE**.

**INPUTS**
  conversation log: /run/media/jun/Jun_6T/Code/TFM_BA_GPT/conversation/interview.jsonl
  spec:             /run/media/jun/Jun_6T/Code/TFM_BA_GPT/conversation/functional_specification.md
  mermaid:          /run/media/jun/Jun_6T/Code/TFM_BA_GPT/conversation/process_diagram.mmd

**GOAL**
  Create: /run/media/jun/Jun_6T/Code/TFM_BA_GPT/experience/<SESSION_ID>/
      interview.jsonl
      functional_specification.md
      process_diagram.mmd
      process_diagram.svg
      process_diagram.png
      process_diagram.pdf            (diagram-only PDF)
      functional_specification.formatted.md
      deliverable.pdf                (final combined PDF)

**STEPS STRICT ORDER**
  1 Read /run/media/jun/Jun_6T/Code/TFM_BA_GPT/conversation/interview.jsonl and extract the first available "session_id".
    If multiple lines: pick the first non-empty "session_id".
    If not found: set SESSION_ID = "sess-unknown".

  2 Create folder /run/media/jun/Jun_6T/Code/TFM_BA_GPT/experience/<SESSION_ID>/

  3 MOVE the three source files into that folder (do not copy) using 'move_file':
      /run/media/jun/Jun_6T/Code/TFM_BA_GPT/conversation/interview.jsonl             -> /run/media/jun/Jun_6T/Code/TFM_BA_GPT/experience/<SESSION_ID>/interview.jsonl
      /run/media/jun/Jun_6T/Code/TFM_BA_GPT/conversation/functional_specification.md -> /run/media/jun/Jun_6T/Code/TFM_BA_GPT/experience/<SESSION_ID>/functional_specification.md
      /run/media/jun/Jun_6T/Code/TFM_BA_GPT/conversation/process_diagram.mmd         -> /run/media/jun/Jun_6T/Code/TFM_BA_GPT/experience/<SESSION_ID>/process_diagram.mmd

  4 Clean the Mermaid file **in place** and report issues (severity/code/message/recommendation):
      clean_mermaid(
        input_path="/run/media/jun/Jun_6T/Code/TFM_BA_GPT/experience/<SESSION_ID>/process_diagram.mmd",
        output_path="/run/media/jun/Jun_6T/Code/TFM_BA_GPT/experience/<SESSION_ID>/process_diagram.mmd",
        overwrite=True
       )

  5 Render Mermaid **SVG + PNG + diagram PDF** in one call:
      render_mermaid_all(
        input_path="/run/media/jun/Jun_6T/Code/TFM_BA_GPT/experience/<SESSION_ID>/process_diagram.mmd",
        output_svg="/run/media/jun/Jun_6T/Code/TFM_BA_GPT/experience/<SESSION_ID>/process_diagram.svg",
        output_png="/run/media/jun/Jun_6T/Code/TFM_BA_GPT/experience/<SESSION_ID>/process_diagram.png",
        output_pdf="/run/media/jun/Jun_6T/Code/TFM_BA_GPT/experience/<SESSION_ID>/process_diagram.pdf",
        theme="forest"
      )
    Confirm the three outputs exist (SVG, PNG, PDF).
    If this tool fails after 2 retries, proceed and try separate renders:
      5.a render_mermaid_svg(...)   (same paths & theme)
      5.b render_mermaid_png(...)   (same paths & theme)
      (diagram PDF is optional; continue without it if needed.)

  6 Format the spec for PDF (prefer **SVG** first):
      format_spec(
        session_dir="/run/media/jun/Jun_6T/Code/TFM_BA_GPT/experience/<SESSION_ID>",
        md="functional_specification.md",
        diagram="process_diagram.svg",
        out="functional_specification.formatted.md"
      )
    Confirm .../functional_specification.formatted.md exists.

  7 Export **deliverable.pdf** (XeLaTeX + fonts):
      export_pdf(
        session_dir="/run/media/jun/Jun_6T/Code/TFM_BA_GPT/experience/<SESSION_ID>",
        md="functional_specification.formatted.md",
        pdf_out="deliverable.pdf",
        engine="xelatex",
        toc=True, toc_depth=2, margin="1in",
        mainfont="Cantarell", monofont="Fira Code"
      )

  8 If Step 7 fails, retry **without custom fonts** (XeLaTeX):
      export_pdf(
        session_dir="/run/media/jun/Jun_6T/Code/TFM_BA_GPT/experience/<SESSION_ID>",
        md="functional_specification.formatted.md",
        pdf_out="deliverable.pdf",
        engine="xelatex",
        toc=True, toc_depth=2, margin="1in"
      )

  9 If still failing, re-format pointing at **PNG** (avoid any SVG issues), then export again:
      format_spec(
        session_dir="/run/media/jun/Jun_6T/Code/TFM_BA_GPT/experience/<SESSION_ID>",
        md="functional_specification.md",
        diagram="process_diagram.png",
        out="functional_specification.formatted.md"
      )
      export_pdf(
        session_dir="/run/media/jun/Jun_6T/Code/TFM_BA_GPT/experience/<SESSION_ID>",
        md="functional_specification.formatted.md",
        pdf_out="deliverable.pdf",
        engine="xelatex",
        toc=True, toc_depth=2, margin="1in"
      )

  10 Final fallback: export with **LuaLaTeX** (no custom fonts):
      export_pdf(
        session_dir="/run/media/jun/Jun_6T/Code/TFM_BA_GPT/experience/<SESSION_ID>",
        md="functional_specification.formatted.md",
        pdf_out="deliverable.pdf",
        engine="lualatex",
        toc=True, toc_depth=2, margin="1in"
      )

  11 Confirm .../experience/<SESSION_ID>/deliverable.pdf exists.

  12 Output exactly: **TERMINATE**

**RULES**
  Be explicit about each step’s success or failure.
  If a tool call fails, retry up to 2 times with the same arguments; if still failing, execute the next fallback step when provided and continue.
"""


In [6]:
termination = TextMentionTermination("TERMINATE")


terminate_agent = AssistantAgent(
    name="terminate_agent",
    model_client=model_client,
    system_message=terminate_system_message,
    tools=(PyMCP_FS_mcp_tools + mermaid_mcp_tools),
    reflect_on_tool_use=False,
)
team = RoundRobinGroupChat(
    [terminate_agent],
    termination_condition=termination,
    )
await Console(team.run_stream(task="Execute the termination pipeline"))

---------- TextMessage (user) ----------
Execute the termination pipeline
---------- ToolCallRequestEvent (terminate_agent) ----------
[FunctionCall(id='call_Yzdk5QcMERDBWgH2YjoqIai3', arguments='{"path":"/run/media/jun/Jun_6T/Code/TFM_BA_GPT/conversation/interview.jsonl"}', name='read_file')]


/run/media/jun/Jun_6T/Code/TFM_BA_GPT/venv/lib/python3.13/site-packages/autogen_agentchat/agents/_assistant_agent.py:1107: UserWarning: Resolved model mismatch: gpt-4o-2024-08-06 != gpt-4o-2024-11-20. Model mapping in autogen_ext.models.openai may be incorrect. Set the model to gpt-4o-2024-11-20 to enhance token/cost estimation and suppress this warning.
  model_result = await model_client.create(


---------- ToolCallExecutionEvent (terminate_agent) ----------
[FunctionExecutionResult(content='[{"type": "text", "text": "{\\"_meta\\": {\\"session_id\\": \\"sess-20231011162620\\", \\"ts\\": \\"2025-08-09T20:35:33.157973Z\\", \\"n_records\\": 2}}\\n{\\"speaker\\": \\"interviewer\\", \\"message\\": \\"QUESTION: Can you briefly describe the new product and its primary purpose?\\", \\"subject\\": \\"Product Overview\\", \\"q_index\\": 1, \\"timestamp\\": \\"2025-08-09T20:35:33.157947Z\\"}\\n{\\"speaker\\": \\"user\\", \\"message\\": \\"The new product is an AI dialer that calls customers to offer travel packages. If the customer accepts, the call transfers to a human. If declined, the AI attempts to convince the customer up to three additional times before ending the call\\", \\"subject\\": \\"Product Overview\\", \\"q_index\\": 1, \\"timestamp\\": \\"2025-08-09T20:35:33.157947Z\\"}\\n{\\"_meta\\": {\\"session_id\\": \\"sess-20231011162620\\", \\"ts\\": \\"2025-08-09T20:36:12.337556Z\\

/run/media/jun/Jun_6T/Code/TFM_BA_GPT/venv/lib/python3.13/site-packages/autogen_agentchat/agents/_assistant_agent.py:1107: UserWarning: Resolved model mismatch: gpt-4o-2024-08-06 != gpt-4o-2024-11-20. Model mapping in autogen_ext.models.openai may be incorrect. Set the model to gpt-4o-2024-11-20 to enhance token/cost estimation and suppress this warning.
  model_result = await model_client.create(


---------- ToolCallExecutionEvent (terminate_agent) ----------
[FunctionExecutionResult(content='[{"type": "text", "text": "Successfully created directory /run/media/jun/Jun_6T/Code/TFM_BA_GPT/experience/sess-20231011162620", "annotations": null}]', name='create_directory', call_id='call_ba4ssEapGczkJfopxqLehKik', is_error=False)]
---------- ToolCallSummaryMessage (terminate_agent) ----------
[{"type": "text", "text": "Successfully created directory /run/media/jun/Jun_6T/Code/TFM_BA_GPT/experience/sess-20231011162620", "annotations": null}]
---------- ToolCallRequestEvent (terminate_agent) ----------
[FunctionCall(id='call_H5MViqGAwtY4I0A5OtfGiEiM', arguments='{"source": "/run/media/jun/Jun_6T/Code/TFM_BA_GPT/conversation/interview.jsonl", "destination": "/run/media/jun/Jun_6T/Code/TFM_BA_GPT/experience/sess-20231011162620/interview.jsonl"}', name='move_file'), FunctionCall(id='call_KIopPTQJ1QuyHXqI3XkTmVU5', arguments='{"source": "/run/media/jun/Jun_6T/Code/TFM_BA_GPT/conversation/fun

/run/media/jun/Jun_6T/Code/TFM_BA_GPT/venv/lib/python3.13/site-packages/autogen_agentchat/agents/_assistant_agent.py:1107: UserWarning: Resolved model mismatch: gpt-4o-2024-08-06 != gpt-4o-2024-11-20. Model mapping in autogen_ext.models.openai may be incorrect. Set the model to gpt-4o-2024-11-20 to enhance token/cost estimation and suppress this warning.
  model_result = await model_client.create(


... (10 outputs left out)

---------- ToolCallExecutionEvent (terminate_agent) ----------
[FunctionExecutionResult(content='[{"type": "text", "text": "size: 742\\ncreated: 2025-08-12 01:36:40.397601\\nmodified: 2025-08-12 01:36:40.397601\\naccessed: 2025-08-12 01:36:40.397601\\nisDirectory: False\\nisFile: True\\npermissions: 644", "annotations": null}]', name='get_file_info', call_id='call_Xzm2UBgwGHShIEQpviIDBmdu', is_error=False)]
---------- ToolCallSummaryMessage (terminate_agent) ----------
[{"type": "text", "text": "size: 742\ncreated: 2025-08-12 01:36:40.397601\nmodified: 2025-08-12 01:36:40.397601\naccessed: 2025-08-12 01:36:40.397601\nisDirectory: False\nisFile: True\npermissions: 644", "annotations": null}]
---------- ToolCallRequestEvent (terminate_agent) ----------
[FunctionCall(id='call_rSNiTQYjNtssrXG1BgtymJXV', arguments='{"path":"/run/media/jun/Jun_6T/Code/TFM_BA_GPT/experience/sess-20231011162620/functional_specification.formatted.md"}', name='write_file')]
---------- ToolCallExecutionEvent (ter

/run/media/jun/Jun_6T/Code/TFM_BA_GPT/venv/lib/python3.13/site-packages/autogen_agentchat/agents/_assistant_agent.py:1107: UserWarning: Resolved model mismatch: gpt-4o-2024-08-06 != gpt-4o-2024-11-20. Model mapping in autogen_ext.models.openai may be incorrect. Set the model to gpt-4o-2024-11-20 to enhance token/cost estimation and suppress this warning.
  model_result = await model_client.create(


---------- TextMessage (terminate_agent) ----------
**TERMINATE**


TaskResult(messages=[TextMessage(id='487d92ad-bb89-4cec-b81a-ea7353ab4fa9', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 8, 11, 23, 35, 36, 391663, tzinfo=datetime.timezone.utc), content='Execute the termination pipeline', type='TextMessage'), ToolCallRequestEvent(id='037baa0e-27ea-41e2-a130-70b71355fdd8', source='terminate_agent', models_usage=RequestUsage(prompt_tokens=3389, completion_tokens=39), metadata={}, created_at=datetime.datetime(2025, 8, 11, 23, 35, 37, 518084, tzinfo=datetime.timezone.utc), content=[FunctionCall(id='call_Yzdk5QcMERDBWgH2YjoqIai3', arguments='{"path":"/run/media/jun/Jun_6T/Code/TFM_BA_GPT/conversation/interview.jsonl"}', name='read_file')], type='ToolCallRequestEvent'), ToolCallExecutionEvent(id='89fcc764-5408-4788-9681-2e751453d3f4', source='terminate_agent', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 8, 11, 23, 35, 37, 819560, tzinfo=datetime.timezone.utc), content=[FunctionExecutionResult(conte